# Hands-on Session 1: Environment Setup and Spatial Data Exploration

**Time:** 10:45-11:05  
**Lead:** Zeyuan

Goals:
- Confirm the Python/Jupyter environment.
- Load DGAT Google Drive `.h5ad` data directly.
- Summarize spot, transcript, and protein matrices with basic statistics.
- Visualize multiple transcript and protein features from the DGAT data.
- Inspect simple spatial neighborhood structure.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

from dgat_tutorial.data import find_dgat_h5ad, load_tutorial_data
from dgat_tutorial.plotting import plot_spatial_feature

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data_dir = repo_root / "data" / "raw"
fig_dir = repo_root / "results" / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {repo_root}")

## Load DGAT Data

This notebook reads the official DGAT Google Drive `.h5ad` data directly, matching the original DGAT workflow. Download the assets with:

```bash
bash scripts/download_dgat_assets.sh
```

The loader searches `external/DGAT_assets/`, `external/DGAT/DGAT_prediction_ST_data/`, and `data/raw/` for `.h5ad` files. If no DGAT file is present, the notebook falls back to a compact synthetic dataset only so participants can still test their environment.

In [ ]:
dgat_h5ad = find_dgat_h5ad(data_dir)
dataset = load_tutorial_data(data_dir, allow_demo=True)
print(f"Loaded DGAT AnnData file: {dgat_h5ad}" if dgat_h5ad else "Loaded synthetic fallback data")

spots = dataset.spots
transcripts = dataset.transcripts
proteins = dataset.proteins

spots.head()

In [ ]:
print("Spots:", spots.shape)
print("Transcript matrix:", transcripts.shape)
print("Protein matrix:", proteins.shape)
print("Coordinate columns present:", {"x", "y"}.issubset(spots.columns))

display(transcripts.iloc[:5, :5])
display(proteins.head())

## Basic Statistics

Before plotting individual markers, summarize the DGAT-provided matrices. These quick checks help participants detect data-loading problems and understand the dynamic range of the dataset.

In [ ]:
transcript_values = transcripts.select_dtypes(include=[np.number])
protein_values = proteins.select_dtypes(include=[np.number])

transcript_library_size = transcript_values.sum(axis=1)
detected_genes = (transcript_values > 0).sum(axis=1)
protein_total = protein_values.sum(axis=1)
detected_proteins = (protein_values > 0).sum(axis=1)

dataset_summary = pd.DataFrame(
    {
        "spots": [len(spots)],
        "genes": [transcript_values.shape[1]],
        "proteins": [protein_values.shape[1]],
        "median_transcript_library_size": [transcript_library_size.median()],
        "median_detected_genes": [detected_genes.median()],
        "median_protein_total": [protein_total.median()],
        "median_detected_proteins": [detected_proteins.median()],
    }
)

display(dataset_summary.round(2))

In [ ]:
def summarize_features(matrix: pd.DataFrame, feature_type: str, n: int = 10) -> pd.DataFrame:
    numeric = matrix.select_dtypes(include=[np.number])
    summary = pd.DataFrame(
        {
            "feature": numeric.columns,
            "feature_type": feature_type,
            "mean": numeric.mean(axis=0).to_numpy(),
            "variance": numeric.var(axis=0).to_numpy(),
            "nonzero_fraction": (numeric > 0).mean(axis=0).to_numpy(),
        }
    )
    return summary.sort_values("variance", ascending=False).head(n)


top_transcript_stats = summarize_features(transcripts, "transcript")
top_protein_stats = summarize_features(proteins, "protein")

display(top_transcript_stats.round(3))
display(top_protein_stats.round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

axes[0].hist(transcript_library_size, bins=30, color="#2f2f2f")
axes[0].set_title("Transcript library size")
axes[0].set_xlabel("Total transcript signal")
axes[0].set_ylabel("Spots")

axes[1].hist(detected_genes, bins=30, color="#5b7f95")
axes[1].set_title("Detected genes per spot")
axes[1].set_xlabel("Genes with value > 0")

axes[2].hist(protein_total, bins=30, color="#b45a3c")
axes[2].set_title("Protein signal per spot")
axes[2].set_xlabel("Total protein signal")

plt.tight_layout()
plt.savefig(fig_dir / "session01_basic_statistics.png", dpi=160)
plt.show()

## Visualize Multiple DGAT Data Features

Rather than hard-coding marker names, select representative high-variance transcript and protein features from the loaded DGAT data. This makes the notebook adapt to the exact files downloaded from Google Drive.

In [ ]:
def top_variable_features(matrix: pd.DataFrame, n: int = 3) -> list[str]:
    numeric = matrix.select_dtypes(include=[np.number])
    return numeric.var(axis=0).sort_values(ascending=False).head(n).index.tolist()


transcript_features = top_variable_features(transcripts, n=3)
protein_features = top_variable_features(proteins, n=3)
features_to_plot = [("Transcript", transcripts, transcript_features, "magma"), ("Protein", proteins, protein_features, "viridis")]

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for row, (label, matrix, features, cmap) in enumerate(features_to_plot):
    for col, feature in enumerate(features):
        plot_spatial_feature(spots, matrix[feature], f"{label}: {feature}", cmap=cmap, ax=axes[row, col])

for ax in axes.ravel():
    ax.tick_params(labelsize=8)

plt.tight_layout()
plt.savefig(fig_dir / "session01_spatial_features.png", dpi=160)
plt.show()

## Explore Spatial Neighborhoods

A common first diagnostic is to ask whether nearby spots have similar molecular profiles. Here we compute nearest neighbors from the spatial coordinates.

In [ ]:
xy = spots[["x", "y"]].to_numpy()
neighbor_count = min(7, len(spots))
distances, neighbor_indices = NearestNeighbors(n_neighbors=neighbor_count).fit(xy).kneighbors(xy)

neighbors = pd.DataFrame(
    {
        "spot_id": spots.index,
        "mean_neighbor_distance": distances[:, 1:].mean(axis=1),
        "nearest_neighbor": spots.index[neighbor_indices[:, 1]],
    }
).set_index("spot_id")

neighbors.head()

In [ ]:
ax = spots.plot.scatter(x="x", y="y", c=neighbors["mean_neighbor_distance"], cmap="cividis", figsize=(5, 4))
ax.set_title("Mean distance to 6 nearest neighbors")
ax.set_aspect("equal")
plt.tight_layout()
plt.savefig(fig_dir / "session01_neighborhood_distances.png", dpi=160)
plt.show()

## Checkpoint

Participants should now know what matrices are available, how spots are arranged spatially, and why neighborhood structure matters for spatial protein inference.